# Part 3: Scalable Processing — Polars Query Optimization, Pandas Benchmark & PySpark Parity
Examines Polars lazy query optimization (predicate pushdown, projection pushdown, parallel window execution), benchmarks execution time and peak RSS against Pandas, and verifies exact parity with PySpark.

In [1]:
from pathlib import Path
print(Path('../outputs/tables/polars_optimized_plan.txt').read_text())

=== POLARS UNOPTIMIZED LOGICAL PLAN ===
FILTER col("texas_ratio") > 1.0
FROM
   WITH_COLUMNS:
   [(col("EQ") / (col("ASSET") + 0.0001)).alias("eq_ratio"), ((col("NCLNLS").fill_null([0.0]) + col("ORE").fill_null([0.0])) / ((col("EQ").fill_null([0.0]) + col("LNATRES").fill_null([0.0])) + 0.0001)).alias("texas_ratio")] 
    SELECT [col("CERT"), col("REPDTE"), col("ASSET"), col("EQ"), col("NCLNLS"), col("ORE"), col("LNATRES")]
      FILTER col("YEAR") >= 2008
      FROM
        Parquet SCAN [/home/henry/Documents/backup/usb/UNI subject/4Term1/Big-data-4-fin/big_data_analysis4Fin/final/data/bronze/financials/YEAR=1984/00000000.parquet, ... 42 other sources]
        PROJECT 33/34 COLUMNS
        ESTIMATED ROWS: 1717671

=== POLARS OPTIMIZED PHYSICAL EXECUTION PLAN ===
FILTER col("texas_ratio") > 1.0
FROM
   WITH_COLUMNS:
   [(col("EQ") / (col("ASSET") + 0.0001)).alias("eq_ratio"), ((col("NCLNLS").fill_null([0.0]) + col("ORE").fill_null([0.0])) / ((col("EQ").fill_null([0.0]) + col("LNATRES").

In [2]:
import polars as pl
print(pl.read_csv('../outputs/tables/part3_benchmark_results.csv'))

scale,rows,engine,time_sec,peak_rss_mb
0.25x (2010-2026),386056,Polars (Lazy),0.158,421.1
0.25x (2010-2026),386056,Polars (Eager),0.346,434.2
0.25x (2010-2026),386056,Polars (Streaming),0.152,446.3
0.25x (2010-2026),386056,Pandas Baseline,1.221,1666.1
0.55x (1995-2026),977097,Polars (Lazy),0.354,560.4
0.55x (1995-2026),977097,Polars (Eager),0.773,578.8
0.55x (1995-2026),977097,Polars (Streaming),0.379,556.8
0.55x (1995-2026),977097,Pandas Baseline,0.995,2075.5
1.00x (1984-2026 Full),1682616,Polars (Lazy),0.572,1327.1
1.00x (1984-2026 Full),1682616,Polars (Eager),1.25,814.9
1.00x (1984-2026 Full),1682616,Polars (Streaming),0.584,1100.8
1.00x (1984-2026 Full),1682616,Pandas Baseline,1.211,2231.1



In [3]:
print(Path('../outputs/tables/spark_polars_parity.json').read_text())
print(Path('../outputs/tables/spark_execution_plan.txt').read_text())

{
  "spark": {
    "row_count": 1682616,
    "mean_eq_ratio": 0.10672,
    "mean_texas_ratio": 279390.22162,
    "mean_netinc_q": 2869.84
  },
  "polars": {
    "row_count": 1682616,
    "mean_eq_ratio": 0.10672,
    "mean_texas_ratio": 279390.22162,
    "mean_netinc_q": 2869.84
  },
  "difference": {
    "row_count_diff": 0,
    "mean_eq_ratio_abs_diff": 0.0,
    "mean_texas_ratio_abs_diff": 0.0,
    "mean_netinc_q_abs_diff": 0.0
  }
}

== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- Window (8)
      +- Sort (7)
         +- Exchange (6)
            +- Project (5)
               +- Window (4)
                  +- Sort (3)
                     +- Exchange (2)
                        +- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [CERT#0, REPDTE#1, ASSET#3, EQ#6, LNLSGR#10, LNATRES#15, NCLNLS#16, ORE#20, NETINC#21, INTINC#22, YEAR#33]
Batched: true
Location: InMemoryFileIndex [file:/home/henry/Documents/backup/usb/UNI subject/4Term1/Big-data-4-fin/big_data_analysis